### Imports

In [8]:
%load_ext autoreload
%autoreload 2

# Internal import
import deep_learning_land_use_classification.config as config
import deep_learning_land_use_classification.evaluation as evaluation
from deep_learning_land_use_classification.dataset import get_multi_label_data

# External imports
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
train_df, test_df, class_names, num_classes = get_multi_label_data()

In [9]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="sstaheli52-wageningen-university-and-research",
    
    # Set the wandb project where this run will be logged.
    project="DL_multi-label-land-use-classification",
    dir = config.WANDB_DIR,

    # Track hyperparameters and run metadata.
    config={
        "architecture": "resnet50",
        "pretrained": True,
        "pretraining_dataset": "ImageNetV2",
        "pretraining_source": "torchvision",
        "weights": "IMAGENET1K_V2",
        "epochs": 5,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "optimizer": "Adam",
        "loss": "BCEWithLogitsLoss",
        "num_classes": num_classes
        },
)

### Resize, transform and normalize dataset

In [10]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # handles inconsistent sizes
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # standard ImageNet mean
        std =[0.229, 0.224, 0.225] # standard ImageNet std
    )
])

### Get training and test dataset, as well as dataset loaders

In [11]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row[self.class_names].values.astype(np.float32))

        return image, label

In [13]:
train_dataset = MultiLabelDataset(train_df, class_names, transform)
test_dataset  = MultiLabelDataset(test_df, class_names, transform)

train_loader = DataLoader(train_dataset, batch_size=run.config.batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=run.config.batch_size, shuffle=False)

### Initiate model

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

Using device: cuda


We use BCEWithLogitsLoss because our goal is multiclass classification. We also penalize larger classes using pos_weight due to the class imbalance present in the dataset. 

In [15]:
labels = train_df[class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=run.config.learning_rate)

In [16]:
run.watch(model, log="all", log_freq=100)

### Train model

In [17]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [18]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
epochs = run.config.epochs

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_loss  = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics_multilabel(
        model,
        test_loader,
        device,
        threshold=0.5
    )

    # Log metrics to Weights & Biases
    run.log({"train_loss": train_loss, "test_loss": test_loss})
    class_metrics = wandb.Table(columns=["class", "precision", "recall", "f1"])
    for i, class_name in enumerate(class_names):
        class_metrics.add_data(class_name, float(precision[i]), float(recall[i]), float(f1[i]))

    run.log({
        "train_loss": train_loss,
        "test_loss": test_loss,
        "macro/precision": p_macro,
        "macro/recall": r_macro,
        "macro/f1": f1_macro,
        "class_metrics": class_metrics,
    })
    
run.finish()
    

Epoch 1/5
Train Loss: 0.7769
Test Loss:  0.4006


### Hyperparameter Tuning using sweep with wandb

In [ ]:
sweep_configuration = {
    "method": "grid",
    "name": "resnet50_epochs_lr_sweep",
    "metric": {
        "goal": "minimize",
        "name": "test_loss",
    },
    "parameters": {
        "epochs": {
            "values": [3, 5, 10],
        },
        "learning_rate": {
            "values": [1e-3, 1e-4, 1e-5],
        },
    },
}

sweep_id = wandb.sweep(
    sweep=sweep_configuration,
    project="DL_land-use-classification",
    entity="sstaheli52-wageningen-university-and-research",
)

# wandb.agent(sweep_id, function=)
# Need to make model training into a function for this to work